In [ ]:
import pandas as pd
import yaml

from src.config import Config
from src.data import load_ihdp, make_ihdp_confounded

In [24]:
df = pd.read_csv("data/ihdp/full/ihdp_full_1.csv")
print(f"Confounder rate: {df['momblack'].mean():.4f}")
df.describe()

Confounder rate: 0.5249


,treat,y_factual,y_cfactual,mu0,mu1,bw,b.head,preterm,birth.o,nnhealth,...,ark,ein,har,mia,pen,tex,was,momwhite,momblack,momhisp
count,985.000000,985.000000,985.000000,985.000000,985.000000,985.000000,985.000000,985.000000,985.000000,985.000000,...,985.000000,985.000000,985.000000,985.000000,985.000000,985.000000,985.000000,985.000000,985.000000,985.000000
mean,0.382741,3.936466,4.853390,2.388199,6.428317,-0.016588,-0.031944,0.004660,0.053872,0.058183,...,0.129949,0.140102,0.140102,0.101523,0.102538,0.139086,0.132995,0.368528,0.524873,0.106599
std,0.486303,2.410530,2.391977,1.269407,0.457758,0.988332,0.985943,0.997664,1.039952,0.992684,...,0.336418,0.347269,0.347269,0.302173,0.303509,0.346212,0.339742,0.482651,0.499635,0.308759
min,0.000000,-1.543902,-1.037628,0.865196,5.525401,-2.731287,-3.800823,-1.850350,-0.879606,-5.255462,...,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,1.902906,2.626934,1.477813,6.060763,-0.677754,-0.602710,-0.733261,-0.879606,-0.441638,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,3.432861,5.579097,2.055528,6.390733,0.122043,0.196818,-0.360898,0.161703,0.183534,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
75%,1.000000,6.184087,6.709119,2.893736,6.732748,0.792143,0.596582,0.756191,0.161703,0.683672,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000
max,1.000000,11.268228,10.171004,9.821792,7.954804,1.505476,2.595403,2.990369,2.244320,2.371637,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [11]:
SPLIT_NAMES = ["train", "val", "test"]

In [23]:
with open("config/ihdp.yaml") as f:
    cfg = Config.model_validate(yaml.safe_load(f))

# Load data and confound it
train_ds, val_ds, test_ds, ytrain_std = load_ihdp(
    cfg.data.path,
    replication=1,
    train_ratio=cfg.data.train_ratio,
    test_ratio=cfg.data.test_ratio,
)
train_ds_conf, val_ds_conf, test_ds_conf = (
    make_ihdp_confounded(ds, effect=cfg.data.confounder_effect)
    for ds in (train_ds, val_ds, test_ds)
)

for split_name, ds in zip(SPLIT_NAMES, (train_ds, val_ds, test_ds), strict=True):
    print(f"{split_name.title()} confounder rate: {ds.confounder.mean():.3f}")

Train confounder rate: 0.541
Val confounder rate: 0.466
Test confounder rate: 0.507


In [ ]:
print("Unconfounded datasets")
for ds, split in zip([train_ds, val_ds, test_ds], SPLIT_NAMES, strict=True):
    print("  " + split)
    print(
        f"    total      -- treatment rate: {ds.a.mean().item():.3f},"
        f" mu0 mean: {ds.mu0.mean().item():.3f},"
        f" mu1 mean: {ds.mu1.mean().item():.3f}"
    )
    for confounder_value in [0, 1]:
        idx = ds.confounder == confounder_value
        print(
            f"    momblack={confounder_value} --"
            f" treatment rate: {ds.a[idx].mean().item():.3f},"
            f" mu0 mean: {ds.mu0[idx].mean().item():.3f},"
            f" mu1 mean: {ds.mu1[idx].mean().item():.3f}"
        )

Unconfounded datasets
  train
    total      -- treatment rate: 0.383, mu0 mean: -0.617, mu1 mean: 1.029
    momblack=0 -- treatment rate: 0.361, mu0 mean: -0.498, mu1 mean: 1.076
    momblack=1 -- treatment rate: 0.402, mu0 mean: -0.717, mu1 mean: 0.989
  val
    total      -- treatment rate: 0.378, mu0 mean: -0.650, mu1 mean: 1.022
    momblack=0 -- treatment rate: 0.392, mu0 mean: -0.581, mu1 mean: 1.052
    momblack=1 -- treatment rate: 0.362, mu0 mean: -0.728, mu1 mean: 0.988
  test
    total      -- treatment rate: 0.385, mu0 mean: -0.679, mu1 mean: 1.011
    momblack=0 -- treatment rate: 0.425, mu0 mean: -0.636, mu1 mean: 1.036
    momblack=1 -- treatment rate: 0.347, mu0 mean: -0.720, mu1 mean: 0.987


In [ ]:
print("Confounded datasets")
for ds, split in zip([train_ds_conf, val_ds_conf, test_ds_conf], SPLIT_NAMES, strict=True):
    print("  " + split)
    print(
        f"    total      -- treatment rate: {ds.a.mean().item():.3f},"
        f" mu0 mean: {ds.mu0.mean().item():.3f},"
        f" mu1 mean: {ds.mu1.mean().item():.3f}"
    )
    for confounder_value in [0, 1]:
        idx = ds.confounder == confounder_value
        print(
            f"    momblack={confounder_value} --"
            f" treatment rate: {ds.a[idx].mean().item():.3f},"
            f" mu0 mean: {ds.mu0[idx].mean().item():.3f},"
            f" mu1 mean: {ds.mu1[idx].mean().item():.3f}"
        )

Confounded datasets
  train
    total      -- treatment rate: 0.489, mu0 mean: -0.617, mu1 mean: 1.029
    momblack=0 -- treatment rate: 0.361, mu0 mean: -0.498, mu1 mean: 1.076
    momblack=1 -- treatment rate: 0.598, mu0 mean: -0.717, mu1 mean: 0.989
  val
    total      -- treatment rate: 0.507, mu0 mean: -0.650, mu1 mean: 1.022
    momblack=0 -- treatment rate: 0.392, mu0 mean: -0.581, mu1 mean: 1.052
    momblack=1 -- treatment rate: 0.638, mu0 mean: -0.728, mu1 mean: 0.988
  test
    total      -- treatment rate: 0.541, mu0 mean: -0.679, mu1 mean: 1.011
    momblack=0 -- treatment rate: 0.425, mu0 mean: -0.636, mu1 mean: 1.036
    momblack=1 -- treatment rate: 0.653, mu0 mean: -0.720, mu1 mean: 0.987
